# Workshop 1 (variant): LoRA Fine-Tuning BERT for Extractive Question Answering

Same LoRA mechanics as `Workshop1_LoRA_Finetuning.ipynb`, but applied to **BERT** on **SQuAD**-style extractive QA instead of Qwen on instruction-following.

**What's different from the causal-LM notebook:**
1. Model class: `AutoModelForCausalLM` -> `AutoModelForQuestionAnswering`
2. Dataset: Alpaca instructions -> SQuAD `(question, context, answer span)`
3. No chat template — question and context are tokenized as a sentence pair
4. Labels are `start_positions` / `end_positions` (token indices), not `input_ids`
5. LoRA `target_modules` use BERT's naming (`query`, `key`, `value`) and `task_type="QUESTION_ANSWERING"`
6. Inference is a single forward pass + argmax over logits, not autoregressive `generate()`

Everything about the early-stopping / loss-curve / adapter-saving plumbing (Sections 8-12) is **identical** to the original notebook — the `Trainer` only cares that the model returns a scalar `loss`, which `AutoModelForQuestionAnswering` does automatically when given `start_positions`/`end_positions`.

## 0. Install dependencies

Same environment as the original workshop notebook (`environment.yml`) — no new packages needed. BERT-base (~110M params) is small enough that we skip 4-bit quantization entirely; LoRA alone is enough to make training fast.

In [ ]:
# Create and activate the conda environment from the YAML file (run once in your terminal):
#
#   conda env create -f environment.yml
#   conda activate 1stWorkshop
#
# Register it as a Jupyter kernel, then select "Python (1stWorkshop)" in VS Code:
#
#   python -m ipykernel install --user --name 1stWorkshop --display-name "Python (1stWorkshop)"

## 1. Imports

`AutoModelForQuestionAnswering` replaces `AutoModelForCausalLM`. We drop `BitsAndBytesConfig` / `prepare_model_for_kbit_training` since BERT-base doesn't need quantization.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, DatasetDict

## 2. Choose the base model

| Model | Size | Notes |
|---|---|---|
| `bert-base-uncased` | 110M | Default — runs on CPU or any GPU |
| `bert-large-uncased` | 340M | Needs a GPU for reasonable speed |
| `distilbert-base-uncased` | 66M | Fastest, slightly lower accuracy |
| `deepset/bert-base-cased-squad2` | 110M | Already SQuAD-tuned — good for comparing against your LoRA run |

In [ ]:
model_name = "bert-base-uncased"

## 3. Load dataset and create train / validation / test splits

SQuAD examples already come as `(question, context, answers)` — there's no prompt-formatting step like `build_messages` in the causal-LM notebook. Each `answers` field looks like:

```python
{"text": ["Paris"], "answer_start": [34]}
```

i.e. the **character offset** of the answer inside `context`. Section 5 converts that character offset into token-index labels.

We take 3000 examples from `rajpurkar/squad`'s train split and make our own 80/10/10 split, instead of using SQuAD's official validation set directly — keeping the same train/val/test workflow as the other notebook. Unlike the causal-LM notebook (which adapts an already-instruction-tuned model), BERT here is learning span-extraction from scratch, so it needs noticeably more examples than 600 to produce sensible answers.

In [ ]:
print("Loading dataset...")

raw_dataset   = load_dataset("rajpurkar/squad")
small_dataset = raw_dataset["train"].shuffle(seed=42).select(range(3000))

# --- Split: 80% train / 10% val / 10% test ---
train_test = small_dataset.train_test_split(test_size=0.20, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.50, seed=42)

data = DatasetDict({
    "train":      train_test["train"],
    "validation": valid_test["train"],
    "test":       valid_test["test"],
})

print(data)
# Expected: ~2400 train / 300 validation / 300 test

In [ ]:
# Quick preview of the first example (raw, before tokenization)
print(small_dataset[0])

## 4. Tokenizer and tokenization

This is the section that differs most from the causal-LM notebook.

- The tokenizer is called with **two arguments** — `question` and `context` — so it inserts BERT's `[SEP]` token between them automatically (no chat template needed).
- `truncation="only_second"` truncates the **context** if the pair is too long, never the question.
- `return_offsets_mapping=True` gives, for every token, the `(char_start, char_end)` span it covers in the original text. We need this to convert the answer's character offset into token indices.
- `sequence_ids(i)` tells us which tokens belong to the question (`0`) vs. the context (`1`), so we know where to search for the answer span and where to mask out non-context tokens.
- If the answer span got truncated out of the context window, both `start_positions` and `end_positions` are set to `0` (the `[CLS]` token), which is the standard SQuAD-preprocessing convention for "no answer in this window".

In [ ]:
print("Preparing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

max_length = 384

def tokenize_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find where the context tokens start/end within this sequence (question=0, context=1)
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context window, label it as unanswerable ([CLS])
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs


tokenized = data.map(tokenize_function, batched=True, remove_columns=data["train"].column_names)
print("Tokenization complete.", tokenized)

## 5. Load the base model

No `BitsAndBytesConfig` / `prepare_model_for_kbit_training` here — at ~110M parameters, BERT-base comfortably fits in full precision on CPU, GPU, or Apple Silicon (MPS), so there's no need for QLoRA's 4-bit tricks (those mattered for the multi-hundred-million/billion-parameter Qwen model).

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
use_cuda = device == "cuda"  # only CUDA gets bf16/fp16 — MPS and CPU stay in fp32

print(f"Loading base model ({device})...")

base_model = AutoModelForQuestionAnswering.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if use_cuda else torch.float32,
)
base_model = base_model.to(device)

print("Base model loaded.")

In [ ]:
for name, _ in base_model.named_modules():
    print(name)

## 6. Attach the LoRA adapter

**`task_type="QUESTION_ANS"`** — this is the swap mentioned in the original notebook's task_type table. Note PEFT's enum value is `QUESTION_ANS`, not the full word `QUESTION_ANSWERING`.

**`target_modules`** — BERT's attention projections are named `query`, `key`, `value` (and the output projection is part of `attention.output.dense`, conventionally skipped for LoRA on BERT). This differs from Qwen's `q_proj`/`k_proj`/`v_proj`/`o_proj` naming — run the cell above (`named_modules`) to confirm the names if you swap to a different encoder model.

In [ ]:
lora_config = LoraConfig(
    task_type="QUESTION_ANS",
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "key", "value"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~0.4-0.6M || all params ~110M || trainable% ~0.4%

## 7. (Optional) MultiEvalTrainer — track test loss as a full curve

Identical to the causal-LM notebook — the `Trainer` only needs the model to return a scalar `loss`, which `AutoModelForQuestionAnswering` computes automatically from `start_positions`/`end_positions`.

In [ ]:
track_test_loss = True

class MultiEvalTrainer(Trainer):
    """Trainer that also evaluates the test set at every eval step."""

    def __init__(self, test_dataset=None, **kwargs):
        super().__init__(**kwargs)
        self._test_dataset = test_dataset

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        metrics = super().evaluate(
            eval_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )
        if eval_dataset is None and self._test_dataset is not None:
            test_metrics = super().evaluate(
                eval_dataset=self._test_dataset,
                ignore_keys=ignore_keys,
                metric_key_prefix="test",
            )
            metrics.update(test_metrics)
        return metrics

## 8. Training arguments and trainer setup

Same early-stopping / gradient-accumulation pattern as the original notebook. Batch size can be larger here since BERT-base + LoRA uses far less memory than 4-bit Qwen.

In [ ]:
os.makedirs("workshop_outputs/lora-bert-qa", exist_ok=True)

training_args = TrainingArguments(
    output_dir="workshop_outputs/lora-bert-qa",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=use_cuda,
    report_to="none",
)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

if track_test_loss:
    trainer = MultiEvalTrainer(test_dataset=tokenized["test"], **trainer_kwargs)
else:
    trainer = Trainer(**trainer_kwargs)

print(f"Trainer ready ({'with' if track_test_loss else 'without'} test loss tracking).")

## 9. Train!

In [ ]:
trainer.train()

## 10. Plot train / validation (/ test) loss curves

Identical plotting code to the causal-LM notebook — it just reads `trainer.state.log_history`, which has the same shape regardless of model/task.

In [ ]:
train_steps, train_losses = [], []
eval_steps,  eval_losses  = [], []
test_steps,  test_losses  = [], []

for entry in trainer.state.log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])
    if "test_loss" in entry:
        test_steps.append(entry["step"])
        test_losses.append(entry["test_loss"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_steps, train_losses, label="Train loss",      color="steelblue",  linewidth=2)
ax.plot(eval_steps,  eval_losses,  label="Validation loss", color="darkorange", linewidth=2, marker="o", markersize=5)
if test_losses:
    ax.plot(test_steps, test_losses, label="Test loss", color="crimson", linewidth=2, marker="s", markersize=5)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("LoRA Fine-Tuning (BERT QA) — Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

plot_path = "workshop_outputs/lora-bert-qa/loss_curves.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Loss plot saved to {plot_path}")

## 11. Save the LoRA adapter

In [ ]:
trainer.save_model("lora_bert_qa_adapter")
tokenizer.save_pretrained("lora_bert_qa_adapter")
print("Saved LoRA adapter to lora_bert_qa_adapter/")

## 12. Reload the adapter and run inference

No `generate()` here — QA is a **single forward pass**. The model outputs `start_logits` and `end_logits` (one score per token for "is this the start/end of the answer"). We search the top-scoring start/end candidates for the best **valid** pair (`start <= end`, within `max_answer_length`) rather than taking independent argmaxes of each — independent argmaxes can pick `end < start`, which silently slices to an empty string.

In [ ]:
base_for_inference = AutoModelForQuestionAnswering.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if use_cuda else torch.float32,
)
base_for_inference = base_for_inference.to(device)

inference_model     = PeftModel.from_pretrained(base_for_inference, "lora_bert_qa_adapter")
inference_tokenizer = AutoTokenizer.from_pretrained("lora_bert_qa_adapter")

test_example = data["test"][0]
question, context = test_example["question"], test_example["context"]

inputs = inference_tokenizer(
    question,
    context,
    max_length=max_length,
    truncation="only_second",
    return_offsets_mapping=True,
    return_tensors="pt",
)
offset_mapping = inputs.pop("offset_mapping")[0]
sequence_ids = inputs.sequence_ids(0)
inputs = {k: v.to(inference_model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = inference_model(**inputs)

start_logits = outputs.start_logits[0]
end_logits   = outputs.end_logits[0]

# Mask out non-context tokens (question + special tokens) so the answer can't be picked from there
context_mask = torch.tensor([sid == 1 for sid in sequence_ids], device=start_logits.device)
start_logits = start_logits.masked_fill(~context_mask, float("-inf"))
end_logits   = end_logits.masked_fill(~context_mask, float("-inf"))

# Taking argmax of start_logits and end_logits independently can pick end < start,
# which silently slices to "". Instead, search the top candidates for the best
# *valid* (start <= end) span - this is the standard SQuAD post-processing approach.
n_best = 20
max_answer_length = 30

start_candidates = torch.topk(start_logits, min(n_best, start_logits.numel())).indices.tolist()
end_candidates   = torch.topk(end_logits, min(n_best, end_logits.numel())).indices.tolist()

best_score = float("-inf")
best_start, best_end = 0, 0
for s in start_candidates:
    for e in end_candidates:
        if e < s or (e - s + 1) > max_answer_length:
            continue
        score = (start_logits[s] + end_logits[e]).item()
        if score > best_score:
            best_score = score
            best_start, best_end = s, e

start_char = offset_mapping[best_start][0].item()
end_char   = offset_mapping[best_end][1].item()
answer = context[start_char:end_char]

print(f"Question: {question}")
print(f"Predicted answer: {answer}")
print(f"Gold answer(s): {test_example['answers']['text']}")

## 13. Extension ideas

- Swap `model_name` to `distilbert-base-uncased` for a faster training loop, or `bert-large-uncased` for higher accuracy
- Compute proper **Exact Match / F1** (the official SQuAD metrics) instead of just eyeballing predictions — `evaluate.load("squad")` from the `evaluate` library does this
- Try `truncation="only_second"` with a sliding window (`return_overflowing_tokens=True`, `stride=128`) to handle contexts longer than `max_length` without losing answers
- Compare this LoRA-tuned BERT against `deepset/bert-base-cased-squad2`, which was fully fine-tuned on the full SQuAD2 dataset
- Merge the adapter permanently: `model.merge_and_unload()` collapses LoRA into the base weights